# Phase 5: Master dataset

In this notebook we combine the results of Phase 3 and Phase 4 into one table. Each row is one neighbourhood, with its car ownership, its nearest station, the station type, the number of disruptions at that station, and the control variables. This table is used for all maps and models.

In [1]:
import os                  # for file paths
import pandas as pd        # for tables

# Folder with the clean data from the earlier phases
clean_folder = 'data/cleaned_data'

# Load the neighbourhoods with their nearest station (Phase 3) and the disruptions per station (Phase 4)
buurten_met_station = pd.read_pickle(os.path.join(clean_folder, 'buurten_met_station.pkl'))
storingen_per_station = pd.read_pickle(os.path.join(clean_folder, 'storingen_per_station.pkl'))

print('buurten_met_station:', len(buurten_met_station), 'rows')
print('storingen_per_station:', len(storingen_per_station), 'rows')

buurten_met_station: 11221 rows
storingen_per_station: 397 rows


## Step 1: Add the disruptions to every neighbourhood

In [2]:
# Every neighbourhood has the code of its nearest station.
# We look up that station code in the disruption table and add its disruption numbers to the neighbourhood.
# Neighbourhoods with the same nearest station get the same disruption numbers.
master = buurten_met_station.merge(storingen_per_station, on='station_code')

# Check: we should still have the same number of neighbourhoods as before
print('Neighbourhoods:', len(master))

# Check: count the empty cells in every column (isna() is True for an empty cell, sum() counts the Trues)
master.isna().sum()

Neighbourhoods: 11221


buurtcode                  0
buurtnaam                  0
gemeente                   0
autos_per_hh               0
inkomen                    0
stedelijkheid              0
hh_grootte                 0
dichtheid                  0
n_huishoudens              0
n_bedrijven                0
afstand_station            0
afstand_overstapstation    0
bedrijven_per_hh           0
station_code               0
station_naam               0
station_type               0
station_groot              0
storingen_per_jaar         0
minuten_per_jaar           0
pct_verstoord              0
treinen_per_dag            0
dtype: int64

## Step 1b: Only neighbourhoods within 10 km of a station


In [3]:
# Feedback: we use a fixed threshold of 10 km around the station.
# For neighbourhoods further away, the station is not really 'nearby', so its type and its disruptions
# say little about car ownership there. With a fixed threshold the results are easier to explain.
# afstand_station is the distance by road (CBS) from the homes in the neighbourhood to the nearest station, in km.
DREMPEL_KM = 10

aantal_voor = len(master)
huishoudens_voor = master['n_huishoudens'].sum()          # all households before the filter

master = master[master['afstand_station'] <= DREMPEL_KM]  # keep only neighbourhoods within 10 km

print('Removed (more than 10 km from a station):', aantal_voor - len(master))
print('Neighbourhoods left:', len(master))

# Share of all households that lives within 10 km of a station
print('Share of households within 10 km:', round(master['n_huishoudens'].sum() / huishoudens_voor * 100, 1), '%')


Removed (more than 10 km from a station): 2182
Neighbourhoods left: 9039
Share of households within 10 km: 86.7 %


## Step 2: Keep only the columns we need

In [4]:
# The columns of the master table, in a logical order
kolommen = [
    'buurtcode', 'buurtnaam', 'gemeente',                              # which neighbourhood
    'autos_per_hh',                                                     # car ownership in that neighbourhood
    'afstand_station', 'afstand_overstapstation',                       # distance to the nearest (transfer) station in km
    'station_code', 'station_naam', 'station_type', 'station_groot',   # which station is nearest, and is it a major station
    'storingen_per_jaar', 'minuten_per_jaar',                           # how often that station has disruptions
    'pct_verstoord', 'treinen_per_dag',                                 # share of disrupted trains (%) and trains per day at that station (2023)
    'inkomen', 'stedelijkheid', 'hh_grootte', 'dichtheid',              # control variables
]
master = master[kolommen]

master.head()

,buurtcode,buurtnaam,gemeente,autos_per_hh,afstand_station,afstand_overstapstation,station_code,station_naam,station_type,station_groot,storingen_per_jaar,minuten_per_jaar,pct_verstoord,treinen_per_dag,inkomen,stedelijkheid,hh_grootte,dichtheid
5,BU16800300,Gasteren,Aa en Hunze,1.5,9.0,9.0,ASN,Assen,knooppuntIntercitystation,True,38.000000,4385.333333,11.237166,155.038356,42.3,5.0,2.2,714.0
6,BU16800400,Anderen,Aa en Hunze,1.8,9.6,9.6,ASN,Assen,knooppuntIntercitystation,True,38.000000,4385.333333,11.237166,155.038356,39.2,5.0,2.3,531.0
10,BU16801100,Annerveenschekanaal,Aa en Hunze,1.4,9.6,20.5,VDM,Veendam,knooppuntStoptreinstation,False,6.333333,1887.333333,3.363901,63.934247,33.3,5.0,2.1,681.0
11,BU16801200,Eexterveenschekanaal,Aa en Hunze,1.5,8.6,22.7,VDM,Veendam,knooppuntStoptreinstation,False,6.333333,1887.333333,3.363901,63.934247,27.9,5.0,2.1,381.0
20,BU16801802,Nieuwediep,Aa en Hunze,1.7,9.2,23.2,VDM,Veendam,knooppuntStoptreinstation,False,6.333333,1887.333333,3.363901,63.934247,30.5,5.0,2.5,256.0


## Step 3: A first look at the numbers

In [5]:
# Summary statistics of every number column: count, mean, standard deviation, minimum, maximum, etc. this was already done in Phase 1, but we do it again to check that the numbers are still the same after merging with the station data.
master.describe()

,autos_per_hh,afstand_station,afstand_overstapstation,storingen_per_jaar,minuten_per_jaar,pct_verstoord,treinen_per_dag,inkomen,stedelijkheid,hh_grootte,dichtheid
count,9039.000000,9039.000000,9039.000000,9039.000000,9039.000000,9039.000000,9039.000000,9039.000000,9039.000000,9039.000000,9039.000000
mean,1.120511,3.802401,10.753844,69.945754,14317.689641,11.175547,173.907589,33.880606,3.117159,2.227846,4333.242726
std,0.364127,2.483615,8.428258,64.839580,14543.169828,4.421645,173.597439,7.864666,1.506015,0.388801,4290.185202
min,0.000000,0.100000,0.100000,2.000000,355.666667,0.261169,0.024658,7.400000,1.000000,1.000000,5.000000
25%,0.900000,1.700000,4.300000,30.333333,4755.000000,7.931381,75.873973,29.200000,2.000000,2.000000,1002.000000
50%,1.100000,3.100000,8.600000,47.666667,10458.333333,10.622805,117.758904,32.800000,3.000000,2.300000,3573.000000
75%,1.400000,5.600000,14.400000,91.333333,16496.333333,13.354971,187.578082,36.900000,5.000000,2.500000,6231.500000
max,2.500000,10.000000,49.000000,535.333333,100046.333333,36.445783,1181.380822,108.000000,5.000000,3.800000,55375.000000


## Saving

In [6]:
# Save the master table: all maps and models (Phase 6 and 7) start from this file
master.to_pickle(os.path.join(clean_folder, 'master.pkl'))